# Week 3: Two-Layer Ensemble

## Strategy

Combine the strengths of two layers:
- **Layer 3**: Best for edge cases (comments, errors, docstrings) - 88.9% edge accuracy
- **Layer 28**: Best overall OOD accuracy - 92.3%

## Logic

```
If Layer 3 is confident it's NOT code (comment/error) → Trust it
Otherwise → Use Layer 28's prediction
```

---

In [ ]:
# Cell 1: Install
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn

In [ ]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm
import warnings
import pickle
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print("Imports ready")

In [ ]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    output_hidden_states=True
)
model.eval()

print(f"Model loaded on {model.device}")

In [ ]:
# Cell 4: Configuration

# The two layers we'll use
EARLY_LAYER = 3   # Best for edge cases (88.9%)
LATE_LAYER = 28   # Best overall (92.3%)

print(f"Two-Layer Ensemble:")
print(f"  Early Layer: {EARLY_LAYER} (edge case detection)")
print(f"  Late Layer:  {LATE_LAYER} (main classification)")

In [ ]:
# Cell 5: Training Data

TRAIN_EXAMPLES = [
    # CODE UNCERTAINTY (label = 1)
    {'prompt': 'import', 'label': 1},
    {'prompt': 'from sklearn import', 'label': 1},
    {'prompt': 'from tensorflow import', 'label': 1},
    {'prompt': 'def process_data(df):\n    df.', 'label': 1},
    {'prompt': 'result = np.', 'label': 1},
    {'prompt': 'model.', 'label': 1},
    {'prompt': 'const [state, setState] = use', 'label': 1},
    {'prompt': 'import { useState } from', 'label': 1},
    {'prompt': 'async function fetch_data() {\n    await', 'label': 1},
    {'prompt': 'model = tf.keras.', 'label': 1},
    {'prompt': 'optimizer = torch.optim.', 'label': 1},
    {'prompt': 'loss = nn.', 'label': 1},
    {'prompt': 'app = FastAPI()\n@app.', 'label': 1},
    {'prompt': '@app.route', 'label': 1},
    {'prompt': 'router.', 'label': 1},
    {'prompt': 'SELECT * FROM users WHERE', 'label': 1},
    {'prompt': 'db.session.', 'label': 1},
    {'prompt': 'git ', 'label': 1},
    {'prompt': 'docker run -', 'label': 1},
    {'prompt': 'npm ', 'label': 1},
    
    # LANGUAGE UNCERTAINTY (label = 0)
    {'prompt': 'This function', 'label': 0},
    {'prompt': 'The algorithm is', 'label': 0},
    {'prompt': 'This code works by', 'label': 0},
    {'prompt': 'The main purpose of', 'label': 0},
    {'prompt': 'Code quality can be', 'label': 0},
    {'prompt': 'The architecture is', 'label': 0},
    {'prompt': 'Performance is', 'label': 0},
    {'prompt': 'Explain what this code', 'label': 0},
    {'prompt': 'First, you need to', 'label': 0},
    {'prompt': 'To implement this,', 'label': 0},
    {'prompt': 'The main advantage of async programming is', 'label': 0},
    {'prompt': 'TypeScript provides better', 'label': 0},
    {'prompt': 'The difference between let and const is', 'label': 0},
    {'prompt': 'Recursion is useful when', 'label': 0},
    {'prompt': 'REST APIs are designed to', 'label': 0},
    {'prompt': 'Unit tests help', 'label': 0},
    {'prompt': 'Returns:', 'label': 0},
    {'prompt': 'Args:', 'label': 0},
    {'prompt': 'Note:', 'label': 0},
    {'prompt': 'Warning:', 'label': 0},
]

print(f"Training: {len(TRAIN_EXAMPLES)} examples")

In [ ]:
# Cell 6: OOD Test Data

OOD_EXAMPLES = [
    # Different languages
    {'prompt': 'fn main() {', 'label': 1, 'cat': 'rust'},
    {'prompt': 'let mut vec = Vec::', 'label': 1, 'cat': 'rust'},
    {'prompt': 'impl Iterator for', 'label': 1, 'cat': 'rust'},
    {'prompt': 'func main() {', 'label': 1, 'cat': 'go'},
    {'prompt': 'package main\nimport', 'label': 1, 'cat': 'go'},
    {'prompt': 'err := http.', 'label': 1, 'cat': 'go'},
    {'prompt': 'defmodule MyApp do', 'label': 1, 'cat': 'elixir'},
    {'prompt': 'def handle_call(', 'label': 1, 'cat': 'elixir'},
    {'prompt': '|> Enum.', 'label': 1, 'cat': 'elixir'},
    {'prompt': 'fun main() =', 'label': 1, 'cat': 'sml'},
    {'prompt': 'let rec fibonacci n =', 'label': 1, 'cat': 'ocaml'},
    {'prompt': 'data Maybe a =', 'label': 1, 'cat': 'haskell'},
    {'prompt': '#include <', 'label': 1, 'cat': 'c'},
    {'prompt': 'int main(int argc,', 'label': 1, 'cat': 'c'},
    {'prompt': 'std::vector<', 'label': 1, 'cat': 'cpp'},
    {'prompt': 'template<typename T>', 'label': 1, 'cat': 'cpp'},
    {'prompt': 'public class Main {', 'label': 1, 'cat': 'java'},
    {'prompt': 'public static void main(String[]', 'label': 1, 'cat': 'java'},
    {'prompt': '@Override\npublic void', 'label': 1, 'cat': 'java'},
    {'prompt': 'interface User {', 'label': 1, 'cat': 'typescript'},
    {'prompt': 'type Props = {', 'label': 1, 'cat': 'typescript'},
    {'prompt': 'const fn: () =>', 'label': 1, 'cat': 'typescript'},
    {'prompt': 'class User < ApplicationRecord', 'label': 1, 'cat': 'ruby'},
    {'prompt': 'def initialize(', 'label': 1, 'cat': 'ruby'},
    {'prompt': '#!/bin/bash\n', 'label': 1, 'cat': 'shell'},
    {'prompt': 'if [ -f', 'label': 1, 'cat': 'shell'},
    
    # Niche libraries
    {'prompt': 'import Bio.', 'label': 1, 'cat': 'niche'},
    {'prompt': 'from scanpy import', 'label': 1, 'cat': 'niche'},
    {'prompt': 'model = xgb.', 'label': 1, 'cat': 'niche'},
    {'prompt': 'client = boto3.', 'label': 1, 'cat': 'niche'},
    {'prompt': 'spark.sql(', 'label': 1, 'cat': 'niche'},
    
    # DevOps
    {'prompt': 'kubectl get', 'label': 1, 'cat': 'devops'},
    {'prompt': 'terraform apply', 'label': 1, 'cat': 'devops'},
    {'prompt': 'ansible-playbook', 'label': 1, 'cat': 'devops'},
    {'prompt': 'aws s3', 'label': 1, 'cat': 'devops'},
    
    # Edge cases - THE KEY TEST
    {'prompt': '# This imports the', 'label': 0, 'cat': 'edge'},
    {'prompt': '// TODO: implement', 'label': 0, 'cat': 'edge'},
    {'prompt': '/* This function', 'label': 0, 'cat': 'edge'},
    {'prompt': '"""This function calculates', 'label': 0, 'cat': 'edge'},
    {'prompt': "'''\nArgs:\n    x:", 'label': 0, 'cat': 'edge'},
    {'prompt': '```python\nimport', 'label': 1, 'cat': 'edge'},
    {'prompt': 'Here is an example:\n```', 'label': 1, 'cat': 'edge'},
    {'prompt': 'ImportError: No module named', 'label': 0, 'cat': 'edge'},
    {'prompt': 'TypeError: expected', 'label': 0, 'cat': 'edge'},
    
    # Language
    {'prompt': 'The architecture of this system', 'label': 0, 'cat': 'lang'},
    {'prompt': 'In conclusion, the results show', 'label': 0, 'cat': 'lang'},
    {'prompt': 'To summarize the key findings', 'label': 0, 'cat': 'lang'},
    {'prompt': 'The primary benefit of using', 'label': 0, 'cat': 'lang'},
    {'prompt': 'This approach is preferred because', 'label': 0, 'cat': 'lang'},
    {'prompt': 'Best practices suggest that', 'label': 0, 'cat': 'lang'},
    {'prompt': 'The main challenge is', 'label': 0, 'cat': 'lang'},
    {'prompt': 'One important consideration is', 'label': 0, 'cat': 'lang'},
]

print(f"OOD Test: {len(OOD_EXAMPLES)} examples")
print(f"  Edge cases: {sum(1 for e in OOD_EXAMPLES if e['cat'] == 'edge')}")

## Extract Hidden States

In [ ]:
# Cell 7: Hidden state extraction

def get_hidden_states(prompt: str, layers: List[int]) -> Dict[int, np.ndarray]:
    """Extract hidden states from specified layers."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    hidden_states = {}
    for layer_idx in layers:
        h = outputs.hidden_states[layer_idx][:, -1, :].squeeze().cpu().numpy()
        hidden_states[layer_idx] = h.astype(np.float32)
    
    return hidden_states

print("Hidden state extraction ready")

In [ ]:
# Cell 8: Extract training data

LAYERS = [EARLY_LAYER, LATE_LAYER]

print(f"Extracting hidden states for layers {LAYERS}...")

train_hidden = {layer: [] for layer in LAYERS}
train_labels = []

for example in tqdm(TRAIN_EXAMPLES, desc="Training"):
    h = get_hidden_states(example['prompt'], LAYERS)
    for layer in LAYERS:
        train_hidden[layer].append(h[layer])
    train_labels.append(example['label'])

for layer in LAYERS:
    train_hidden[layer] = np.array(train_hidden[layer])
train_labels = np.array(train_labels)

print(f"Done! Shape: {train_hidden[EARLY_LAYER].shape}")

In [ ]:
# Cell 9: Extract OOD data

print("Extracting OOD hidden states...")

ood_hidden = {layer: [] for layer in LAYERS}
ood_labels = []
ood_categories = []

for example in tqdm(OOD_EXAMPLES, desc="OOD"):
    h = get_hidden_states(example['prompt'], LAYERS)
    for layer in LAYERS:
        ood_hidden[layer].append(h[layer])
    ood_labels.append(example['label'])
    ood_categories.append(example['cat'])

for layer in LAYERS:
    ood_hidden[layer] = np.array(ood_hidden[layer])
ood_labels = np.array(ood_labels)

print(f"Done!")

## Train Individual Probes

In [ ]:
# Cell 10: Train probes for each layer

print("Training individual probes...")
print("="*60)

probes = {}
scalers = {}

for layer in LAYERS:
    X = train_hidden[layer]
    y = train_labels
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    probe = LogisticRegression(max_iter=1000, random_state=42)
    
    # LOO CV
    loo = LeaveOneOut()
    y_pred = cross_val_predict(probe, X_scaled, y, cv=loo)
    train_acc = accuracy_score(y, y_pred)
    
    # Train final
    probe.fit(X_scaled, y)
    
    # OOD eval
    X_ood = ood_hidden[layer]
    X_ood_scaled = scaler.transform(X_ood)
    ood_acc = accuracy_score(ood_labels, probe.predict(X_ood_scaled))
    
    probes[layer] = probe
    scalers[layer] = scaler
    
    print(f"Layer {layer}: Train={train_acc:.1%}, OOD={ood_acc:.1%}")

print("\nProbes trained!")

## Two-Layer Ensemble

In [ ]:
# Cell 11: Define the Two-Layer Ensemble

class TwoLayerEnsemble:
    """
    Two-layer ensemble that uses:
    - Early layer to detect edge cases (comments, errors)
    - Late layer for main classification
    
    Logic:
    - If early layer is confident it's NOT code → trust it (likely comment/error)
    - Otherwise → use late layer's prediction
    """
    
    def __init__(self, 
                 early_probe, early_scaler, early_layer: int,
                 late_probe, late_scaler, late_layer: int,
                 early_threshold: float = 0.3):
        self.early_probe = early_probe
        self.early_scaler = early_scaler
        self.early_layer = early_layer
        
        self.late_probe = late_probe
        self.late_scaler = late_scaler
        self.late_layer = late_layer
        
        self.early_threshold = early_threshold
    
    def predict_single(self, hidden_states: Dict[int, np.ndarray]) -> Tuple[int, float, str]:
        """Predict for a single example."""
        # Get early layer probability
        h_early = hidden_states[self.early_layer].reshape(1, -1)
        h_early_scaled = self.early_scaler.transform(h_early)
        p_early = self.early_probe.predict_proba(h_early_scaled)[0, 1]
        
        # Get late layer probability
        h_late = hidden_states[self.late_layer].reshape(1, -1)
        h_late_scaled = self.late_scaler.transform(h_late)
        p_late = self.late_probe.predict_proba(h_late_scaled)[0, 1]
        
        # Decision logic
        if p_early < self.early_threshold:
            # Early layer confident it's NOT code (comment/error/docstring)
            return 0, p_early, "early_override"
        else:
            # Use late layer
            pred = 1 if p_late > 0.5 else 0
            return pred, p_late, "late_layer"
    
    def predict(self, hidden_states_list: List[Dict[int, np.ndarray]]) -> List[Tuple[int, float, str]]:
        """Predict for multiple examples."""
        return [self.predict_single(h) for h in hidden_states_list]

print("TwoLayerEnsemble defined")

In [ ]:
# Cell 12: Test different thresholds

print("Testing different early_threshold values...")
print("="*60)

# Prepare hidden states list for OOD
ood_hidden_list = [
    {layer: ood_hidden[layer][i] for layer in LAYERS}
    for i in range(len(OOD_EXAMPLES))
]

threshold_results = []

for threshold in [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
    ensemble = TwoLayerEnsemble(
        early_probe=probes[EARLY_LAYER],
        early_scaler=scalers[EARLY_LAYER],
        early_layer=EARLY_LAYER,
        late_probe=probes[LATE_LAYER],
        late_scaler=scalers[LATE_LAYER],
        late_layer=LATE_LAYER,
        early_threshold=threshold
    )
    
    predictions = ensemble.predict(ood_hidden_list)
    preds = [p[0] for p in predictions]
    sources = [p[2] for p in predictions]
    
    acc = accuracy_score(ood_labels, preds)
    early_overrides = sum(1 for s in sources if s == "early_override")
    
    # Edge case accuracy
    edge_indices = [i for i, e in enumerate(OOD_EXAMPLES) if e['cat'] == 'edge']
    edge_preds = [preds[i] for i in edge_indices]
    edge_labels = [ood_labels[i] for i in edge_indices]
    edge_acc = accuracy_score(edge_labels, edge_preds)
    
    threshold_results.append({
        'threshold': threshold,
        'accuracy': acc,
        'edge_accuracy': edge_acc,
        'early_overrides': early_overrides,
    })
    
    print(f"Threshold {threshold:.2f}: OOD={acc:.1%}, Edge={edge_acc:.1%}, Overrides={early_overrides}")

# Find best
best_thresh = max(threshold_results, key=lambda x: x['accuracy'])
print(f"\nBest threshold: {best_thresh['threshold']:.2f} with {best_thresh['accuracy']:.1%} OOD accuracy")

In [ ]:
# Cell 13: Create final ensemble with best threshold

BEST_THRESHOLD = best_thresh['threshold']

final_ensemble = TwoLayerEnsemble(
    early_probe=probes[EARLY_LAYER],
    early_scaler=scalers[EARLY_LAYER],
    early_layer=EARLY_LAYER,
    late_probe=probes[LATE_LAYER],
    late_scaler=scalers[LATE_LAYER],
    late_layer=LATE_LAYER,
    early_threshold=BEST_THRESHOLD
)

print(f"Final ensemble created with threshold={BEST_THRESHOLD}")

In [ ]:
# Cell 14: Detailed evaluation

print("\n" + "="*80)
print("DETAILED EVALUATION")
print("="*80)

# Get predictions
predictions = final_ensemble.predict(ood_hidden_list)
preds = [p[0] for p in predictions]
probs = [p[1] for p in predictions]
sources = [p[2] for p in predictions]

# Overall accuracy
overall_acc = accuracy_score(ood_labels, preds)
print(f"\nOverall OOD Accuracy: {overall_acc:.1%} ({sum(p == l for p, l in zip(preds, ood_labels))}/{len(ood_labels)})")

# Compare to single layers
late_only_preds = probes[LATE_LAYER].predict(scalers[LATE_LAYER].transform(ood_hidden[LATE_LAYER]))
late_only_acc = accuracy_score(ood_labels, late_only_preds)

print(f"\nComparison:")
print(f"  Layer {LATE_LAYER} only:    {late_only_acc:.1%}")
print(f"  Two-Layer Ensemble: {overall_acc:.1%}")
print(f"  Improvement:        {(overall_acc - late_only_acc)*100:+.1f}%")

# By category
print(f"\nBy Category:")
for cat in sorted(set(ood_categories)):
    indices = [i for i, c in enumerate(ood_categories) if c == cat]
    cat_preds = [preds[i] for i in indices]
    cat_labels = [ood_labels[i] for i in indices]
    cat_sources = [sources[i] for i in indices]
    
    cat_acc = accuracy_score(cat_labels, cat_preds)
    early_count = sum(1 for s in cat_sources if s == "early_override")
    
    # Compare to late-only
    late_cat_preds = [late_only_preds[i] for i in indices]
    late_cat_acc = accuracy_score(cat_labels, late_cat_preds)
    
    diff = cat_acc - late_cat_acc
    diff_str = f"{diff*100:+.0f}%" if diff != 0 else "="
    
    print(f"  {cat:<12} {cat_acc:.0%} ({len(indices):>2} examples, {early_count} overrides) vs late-only {late_cat_acc:.0%} [{diff_str}]")

In [ ]:
# Cell 15: Error analysis

print("\n" + "="*80)
print("ERROR ANALYSIS")
print("="*80)

errors = []
for i, (example, pred, prob, source) in enumerate(zip(OOD_EXAMPLES, preds, probs, sources)):
    if pred != example['label']:
        errors.append({
            'prompt': example['prompt'],
            'category': example['cat'],
            'true': 'CODE' if example['label'] == 1 else 'LANG',
            'pred': 'CODE' if pred == 1 else 'LANG',
            'prob': prob,
            'source': source,
        })

print(f"\nTotal errors: {len(errors)}/{len(OOD_EXAMPLES)}")
print("-"*60)

for e in errors:
    print(f"  [{e['category']}] '{e['prompt'][:40]}...'")
    print(f"    True: {e['true']}, Pred: {e['pred']}, P={e['prob']:.2f}, Source: {e['source']}")
    print()

# Compare to late-only errors
late_errors = sum(1 for p, l in zip(late_only_preds, ood_labels) if p != l)
print(f"Late-only errors: {late_errors}")
print(f"Ensemble errors:  {len(errors)}")
print(f"Errors saved:     {late_errors - len(errors)}")

In [ ]:
# Cell 16: Visualize threshold effects

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

thresholds = [r['threshold'] for r in threshold_results]
accuracies = [r['accuracy'] for r in threshold_results]
edge_accs = [r['edge_accuracy'] for r in threshold_results]
overrides = [r['early_overrides'] for r in threshold_results]

# Plot 1: Accuracy vs Threshold
ax1 = axes[0]
ax1.plot(thresholds, accuracies, 'o-', color='coral', linewidth=2, markersize=8, label='Overall OOD')
ax1.plot(thresholds, edge_accs, 's-', color='steelblue', linewidth=2, markersize=8, label='Edge Cases')
ax1.axhline(y=late_only_acc, color='gray', linestyle='--', label=f'Layer {LATE_LAYER} only ({late_only_acc:.1%})')
ax1.axvline(x=BEST_THRESHOLD, color='green', linestyle=':', alpha=0.5)
ax1.set_xlabel('Early Layer Threshold')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy vs Early Threshold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Number of overrides
ax2 = axes[1]
ax2.bar(thresholds, overrides, color='purple', alpha=0.7, width=0.04)
ax2.axvline(x=BEST_THRESHOLD, color='green', linestyle=':', alpha=0.5, label=f'Best threshold')
ax2.set_xlabel('Early Layer Threshold')
ax2.set_ylabel('Number of Early Overrides')
ax2.set_title('How Often Early Layer Overrides')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('week3_twolayer_ensemble.png', dpi=150)
plt.show()

In [ ]:
# Cell 17: Save the ensemble

print("\n" + "="*60)
print("SAVING ENSEMBLE")
print("="*60)

ensemble_data = {
    'early_layer': EARLY_LAYER,
    'late_layer': LATE_LAYER,
    'early_probe': probes[EARLY_LAYER],
    'late_probe': probes[LATE_LAYER],
    'early_scaler': scalers[EARLY_LAYER],
    'late_scaler': scalers[LATE_LAYER],
    'threshold': BEST_THRESHOLD,
    'ood_accuracy': overall_acc,
}

with open('twolayer_ensemble.pkl', 'wb') as f:
    pickle.dump(ensemble_data, f)

print(f"\nSaved: twolayer_ensemble.pkl")
print(f"  Early layer: {EARLY_LAYER}")
print(f"  Late layer:  {LATE_LAYER}")
print(f"  Threshold:   {BEST_THRESHOLD}")
print(f"  OOD Accuracy: {overall_acc:.1%}")

In [ ]:
# Cell 18: Final Summary

print("\n" + "="*80)
print("FINAL SUMMARY: TWO-LAYER ENSEMBLE")
print("="*80)

print(f"""
ARCHITECTURE:
────────────────────────────────────────────────────────────────────────
  Early Layer (L{EARLY_LAYER}):  Detects edge cases (comments, errors, docstrings)
  Late Layer (L{LATE_LAYER}):   Main classification (code vs language)
  Threshold:        {BEST_THRESHOLD} (if P_early < threshold → override)

LOGIC:
────────────────────────────────────────────────────────────────────────
  if early_layer_prob < {BEST_THRESHOLD}:
      return LANGUAGE  # Early layer confident it's a comment/error
  else:
      return late_layer_prediction  # Use Layer {LATE_LAYER}

RESULTS:
────────────────────────────────────────────────────────────────────────
  Layer {LATE_LAYER} alone:    {late_only_acc:.1%} ({int(late_only_acc * len(ood_labels))}/{len(ood_labels)} correct)
  Two-Layer Ensemble: {overall_acc:.1%} ({int(overall_acc * len(ood_labels))}/{len(ood_labels)} correct)
  
  Improvement:        {(overall_acc - late_only_acc)*100:+.1f}%
  Errors reduced:     {late_errors} → {len(errors)}

EDGE CASE PERFORMANCE:
────────────────────────────────────────────────────────────────────────
  Layer {LATE_LAYER} on edges:  {accuracy_score([ood_labels[i] for i in [j for j, e in enumerate(OOD_EXAMPLES) if e['cat'] == 'edge']], [late_only_preds[i] for i in [j for j, e in enumerate(OOD_EXAMPLES) if e['cat'] == 'edge']]):.1%}
  Ensemble on edges:  {best_thresh['edge_accuracy']:.1%}
""")

print("="*80)